Expandir `docker-compose.yml` **de forma incremental**, manteniendo lo que ya funciona y añadiendo los módulos DevOps (**ELK**, **Prometheus/Grafana**, **Microservicios**) con enfoque en **producción local**. Incluiré:

1. **Estructura de archivos necesarios**.
2. **Configuraciones seguras para producción** (aunque sea localhost).
3. **Checks de validación y errores comunes**.
4. **Integración sin romper tu stack actual**.

---

## **Paso 1: ELK Stack (Log Management)**
### **Objetivo**:  
Centralizar logs del backend (Node.js), frontend (Nginx/PHP), Avalanche y Vault. Almacenamiento y Búsqueda de Logs

---
---

### <font color="green">**Elasticsearch: Almacenamiento y Búsqueda de Logs**:</font> 
- Almacena e indexa todos los logs del sistema

- Permite búsquedas rápidas y eficientes

- Escalable para manejar grandes volúmenes de datos

### <font color="green">**Logstash: Procesamiento de Logs**: </font> 
- Interfaz web para visualizar y analizar logs

- Creación de dashboards personalizados

- Búsqueda avanzada y filtrado de logs

### <font color="green">**Kibana: Visualización de Logs**: </font> 
- Recoge logs de múltiples fuentes (backend, frontend, nginx)

- Filtra y transforma los datos

- Envía los logs procesados a Elasticsearch

---
---


### **1. Añade estos servicios al `docker-compose.yml`**:
```yaml
services:
  elasticsearch:
    image: docker.elastic.co/elasticsearch/elasticsearch:8.12.0
    container_name: elasticsearch
    environment:
      - discovery.type=single-node
      - xpack.security.enabled=false  # Desactivar seguridad para desarrollo local
      - ES_JAVA_OPTS=-Xms1g -Xmx1g    # Ajustar memoria según tu sistema
    volumes:
      - es_data:/usr/share/elasticsearch/data
    ports:
      - "9200:9200"
    networks:
      - transcendence
    restart: unless-stopped

  logstash:
    image: docker.elastic.co/logstash/logstash:8.12.0
    container_name: logstash
    volumes:
      - ./elk/logstash/pipeline:/usr/share/logstash/pipeline  # Configuración de pipelines
      - ./elk/logstash/config:/usr/share/logstash/config      # Configuración adicional
    environment:
      - LS_JAVA_OPTS=-Xms512m -Xmx512m
    depends_on:
      - elasticsearch
    networks:
      - transcendence
    restart: unless-stopped

  kibana:
    image: docker.elastic.co/kibana/kibana:8.12.0
    container_name: kibana
    ports:
      - "5601:5601"
    depends_on:
      - elasticsearch
    networks:
      - transcendence
    restart: unless-stopped

volumes:
  es_data:
    driver: local
    driver_opts:
      type: none
      device: "$HOME/goinfre/data/elasticsearch"  # Persistencia en Mac (ajusta para Linux/Windows)
      o: bind
```

### **2. Archivos necesarios**:
- **`./elk/logstash/pipeline/logstash.conf`** (Configuración de entrada/salida):
  ```conf
  input {
    file {
      path => "/var/log/backend.log"
      start_position => "beginning"
    }
    file {
      path => "/var/log/nginx/access.log"
      type => "nginx"
    }
  }
  filter {
    if [type] == "nginx" {
      grok {
        match => { "message" => "%{COMBINEDAPACHELOG}" }
      }
    }
  }
  output {
    elasticsearch {
      hosts => ["http://elasticsearch:9200"]
      index => "logs-%{+YYYY.MM.dd}"
    }
  }
  ```
- **`./elk/logstash/config/logstash.yml`** (Opcional):
  ```yaml
  http.host: "0.0.0.0"
  ```

### **3. Integración con tus servicios existentes**:
- **Backend (Node.js)**:  
  Asegúrate de que tu aplicación redirija logs a un archivo (ej: `/var/log/backend.log`). Usa `winston` o `morgan`:
  ```javascript
  // En tu backend (app.js)
  const fs = require('fs');
  const morgan = require('morgan');
  const logStream = fs.createWriteStream('/var/log/backend.log', { flags: 'a' });
  app.use(morgan('combined', { stream: logStream }));
  ```

- **Nginx/PHP**:  
  Configura Nginx para enviar logs a Logstash:
  ```nginx
  # En tu configuración de Nginx (php service)
  access_log /var/log/nginx/access.log;
  error_log /var/log/nginx/error.log;
  ```

### **4. Validación y errores comunes**:
- **Check 1**: Verifica que Elasticsearch esté accesible en `http://localhost:9200`.
- **Check 2**: Kibana debe responder en `http://localhost:5601`.
- **Error común**:  
  - *"Logstash no envía logs a Elasticsearch"* → Revisa el archivo `logstash.conf` y los permisos de los archivos de log.
  - *"Out of memory"* → Ajusta `ES_JAVA_OPTS` y `LS_JAVA_OPTS` en los servicios.

---

## **Paso 2: Prometheus + Grafana (Monitoring)**
### **Objetivo**:  
Monitorear métricas de Node.js, Nginx, Avalanche y sistema.

### **1. Añade estos servicios al `docker-compose.yml`**:
```yaml
services:
  prometheus:
    image: prom/prometheus
    container_name: prometheus
    volumes:
      - ./monitoring/prometheus.yml:/etc/prometheus/prometheus.yml
    ports:
      - "9090:9090"
    networks:
      - transcendence
    restart: unless-stopped

  grafana:
    image: grafana/grafana
    container_name: grafana
    volumes:
      - grafana_data:/var/lib/grafana
    ports:
      - "3003:3000"  # Grafana usa el puerto 3000 por defecto
    depends_on:
      - prometheus
    networks:
      - transcendence
    restart: unless-stopped

volumes:
  grafana_data:
    driver: local
    driver_opts:
      type: none
      device: "$HOME/goinfre/data/grafana"
      o: bind
```

### **2. Archivos necesarios**:
- **`./monitoring/prometheus.yml`**:
  ```yaml
  global:
    scrape_interval: 15s

  scrape_configs:
    - job_name: "nodejs"
      static_configs:
        - targets: ["backend:3000"]  # Métricas de Node.js (usa 'prom-client')
    - job_name: "nginx"
      metrics_path: "/nginx-metrics"
      static_configs:
        - targets: ["php:8080"]      # Métricas de Nginx (requiere módulo 'nginx-exporter')
    - job_name: "avalanche"
      static_configs:
        - targets: ["avalanche:9650"] # Métricas de Avalanche (si expone /metrics)
  ```

### **3. Integración con tus servicios**:
- **Backend (Node.js)**:  
  Instala `prom-client` para exponer métricas:
  ```bash
  npm install prom-client
  ```
  ```javascript
  // En tu backend (app.js)
  const client = require('prom-client');
  const collectDefaultMetrics = client.collectDefaultMetrics;
  collectDefaultMetrics({ timeout: 5000 });

  app.get('/metrics', async (req, res) => {
    res.set('Content-Type', client.register.contentType);
    res.end(await client.register.metrics());
  });
  ```

- **Nginx**:  
  Usa `nginx-exporter` o configura stub_status:
  ```nginx
  server {
    location /nginx-metrics {
      stub_status on;
      allow 127.0.0.1;
      deny all;
    }
  }
  ```

### **4. Validación y errores comunes**:
- **Check 1**: Accede a Prometheus en `http://localhost:9090/targets`. Todos los `targets` deben estar "UP".
- **Check 2**: Grafana en `http://localhost:3003` (usuario: `admin`, contraseña: `admin`).
- **Error común**:  
  - *"Prometheus no scrapea métricas"* → Revisa los `targets` en `prometheus.yml` y la conectividad entre contenedores.

---

## **Paso 3: Microservicios (Backend Reestructurado)**
### **Objetivo**:  
Dividir el backend monolítico en servicios independientes (ej: `auth`, `users`, `blockchain`).

### **1. Estructura propuesta**:
```
./microservices/
├── auth/
│   ├── Dockerfile
│   ├── package.json
│   └── src/ (código)
├── users/
│   ├── Dockerfile
│   └── ...
└── blockchain/
    ├── Dockerfile
    └── ...
```

### **2. Ajustes en `docker-compose.yml`**:
Reemplaza el servicio `backend` por:
```yaml
services:
  auth-service:
    build: ./microservices/auth
    ports:
      - "3002:3000"
    networks:
      - transcendence
    depends_on:
      - sqlite

  user-service:
    build: ./microservices/users
    ports:
      - "3003:3000"
    networks:
      - transcendence

  blockchain-service:
    build: ./microservices/blockchain
    ports:
      - "3004:3000"
    networks:
      - transcendence
```

### **3. Comunicación entre microservicios**:
- Usa **REST APIs** (HTTP) o **message queues** (RabbitMQ).
- Ejemplo con RabbitMQ:
  ```yaml
  rabbitmq:
    image: rabbitmq:3-management
    ports:
      - "5672:5672"  # AMQP
      - "15672:15672"  # Management UI
    networks:
      - transcendence
  ```

### **4. Validación y errores comunes**:
- **Check 1**: Cada microservicio debe responder en su puerto (ej: `http://localhost:3002/health`).
- **Error común**:  
  - *"CORS o conexión rechazada"* → Configura CORS en cada servicio y verifica la red en Docker.

---

## **Resumen de Pasos**
1. **ELK**:  
   - Añade servicios + configura Logstash.  
   - Integra logs de Node.js y Nginx.  
2. **Prometheus/Grafana**:  
   - Configura scraping de métricas.  
   - Expón `/metrics` en Node.js.  
3. **Microservicios**:  
   - Divide el backend y usa RabbitMQ para comunicación.  

### **Próximos pasos**:
- Profundizar en alguno de estos módulos
- **tests** o **errores específicos**

Todo está diseñado para **producción local** (volúmenes persistentes, reinicios automáticos, seguridad básica).